# PyArrow File Viewer

A utility notebook to view and inspect PyArrow/Arrow files.

Supports:
- Arrow IPC (Feather) files
- Parquet files  
- HuggingFace datasets (backed by Arrow)

## Usage

1. Set the `file_path` variable in the "View Arrow File" section
2. Run the cells to view file information, schema, and sample data
3. Optionally enable statistics for numeric columns

In [2]:
# Imports
import sys
from pathlib import Path
from typing import Any

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pyarrow.ipc as ipc
    print("✓ PyArrow imported successfully")
except ImportError:
    print("Error: pyarrow is not installed. Please install it with: pip install pyarrow")
    sys.exit(1)

try:
    from datasets import load_from_disk, Dataset
    print("✓ HuggingFace datasets imported successfully")
except ImportError:
    print("⚠ HuggingFace datasets not available (optional)")
    Dataset = None
    load_from_disk = None

import numpy as np

✓ PyArrow imported successfully


/home/khw/miniconda3/envs/alohax_lerobot_fork/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ HuggingFace datasets imported successfully


In [3]:
# Helper Functions

def format_size(size_bytes: int) -> str:
    """Format bytes to human-readable size."""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.2f} PB"


def print_schema(schema: pa.Schema, indent: int = 0) -> None:
    """Print Arrow schema in a readable format."""
    prefix = "  " * indent
    print(f"{prefix}Schema:")
    for i, field in enumerate(schema):
        field_type = str(field.type)
        nullable = "nullable" if field.nullable else "non-nullable"
        print(f"{prefix}  [{i}] {field.name}: {field_type} ({nullable})")
        if hasattr(field.type, 'value_type'):
            print(f"{prefix}      value_type: {field.type.value_type}")


def print_table_info(table: pa.Table, max_rows: int = 10) -> None:
    """Print information about an Arrow table."""
    print("\n" + "=" * 80)
    print("TABLE INFORMATION")
    print("=" * 80)
    print(f"Number of rows: {len(table):,}")
    print(f"Number of columns: {len(table.column_names)}")
    print(f"Total size: {format_size(table.nbytes)}")
    
    print("\n" + "-" * 80)
    print("SCHEMA")
    print("-" * 80)
    print_schema(table.schema)
    
    print("\n" + "-" * 80)
    print(f"COLUMN NAMES ({len(table.column_names)} columns)")
    print("-" * 80)
    for i, name in enumerate(table.column_names):
        col = table[name]
        dtype = str(col.type)
        null_count = col.null_count
        print(f"  [{i}] {name}: {dtype} (nulls: {null_count}/{len(col)})")
    
    if max_rows > 0:
        print("\n" + "-" * 80)
        print(f"FIRST {min(max_rows, len(table))} ROWS (sample)")
        print("-" * 80)
        sample_table = table.slice(0, min(max_rows, len(table)))
        
        # Print column headers
        headers = table.column_names
        print(" | ".join(f"{h[:20]:<20}" for h in headers))
        print("-" * 80)
        
        # Print sample rows
        for i in range(min(max_rows, len(table))):
            row_values = []
            for col_name in headers:
                col = table[col_name]
                val = col[i].as_py()
                if isinstance(val, (list, dict)):
                    val_str = str(val)[:50] + "..." if len(str(val)) > 50 else str(val)
                elif isinstance(val, bytes):
                    val_str = f"<bytes: {len(val)} bytes>"
                else:
                    val_str = str(val)[:20]
                row_values.append(f"{val_str:<20}")
            print(" | ".join(row_values))


def print_statistics(table: pa.Table) -> None:
    """Print statistics about numeric columns in the table."""
    print("\n" + "=" * 80)
    print("STATISTICS (numeric columns only)")
    print("=" * 80)
    
    for col_name in table.column_names:
        col = table[col_name]
        if pa.types.is_numeric(col.type):
            try:
                arr = col.to_numpy()
                arr = arr[~np.isnan(arr)] if pa.types.is_floating(col.type) else arr
                
                if len(arr) > 0:
                    print(f"\n{col_name}:")
                    print(f"  Type: {col.type}")
                    print(f"  Count: {len(arr):,}")
                    print(f"  Nulls: {col.null_count}")
                    if len(arr) > 0:
                        print(f"  Min: {np.min(arr)}")
                        print(f"  Max: {np.max(arr)}")
                        print(f"  Mean: {np.mean(arr):.4f}")
                        print(f"  Std: {np.std(arr):.4f}")
            except Exception as e:
                print(f"\n{col_name}: Could not compute statistics ({e})")

In [4]:
# Main Viewing Functions

def view_arrow_file(file_path: Path, max_rows: int = 10, show_stats: bool = False) -> pa.Table | None:
    """View a single Arrow file. Returns the Arrow table if successful."""
    print(f"\n{'=' * 80}")
    print(f"VIEWING ARROW FILE: {file_path}")
    print(f"{'=' * 80}")
    
    if not file_path.exists():
        print(f"Error: File not found: {file_path}")
        return None
    
    file_size = file_path.stat().st_size
    print(f"File size: {format_size(file_size)}")
    
    try:
        # Try to open as Arrow IPC file
        with pa.ipc.open_file(file_path) as reader:
            print("\nFile type: Arrow IPC (Feather) format")
            schema = reader.schema
            num_batches = reader.num_record_batches
            
            print(f"\nNumber of record batches: {num_batches}")
            print_schema(schema)
            
            if num_batches > 0:
                # Read all batches and combine into a table
                batches = [reader.get_batch(i) for i in range(num_batches)]
                table = pa.Table.from_batches(batches)
                print_table_info(table, max_rows=max_rows)
                
                if show_stats:
                    print_statistics(table)
                
                return table
    
    except (pa.lib.ArrowInvalid, pa.lib.ArrowIOError):
        # Try as Parquet file
        try:
            print("\nFile type: Parquet format")
            parquet_file = pq.ParquetFile(file_path)
            
            print(f"\nNumber of row groups: {parquet_file.num_row_groups}")
            print_schema(parquet_file.schema)
            
            # Read the entire file
            table = parquet_file.read()
            print_table_info(table, max_rows=max_rows)
            
            if show_stats:
                print_statistics(table)
            
            return table
        
        except Exception as e:
            print(f"\nError reading file: {e}")
            print("Trying alternative methods...")
            
            # Try reading as raw Arrow file
            try:
                with open(file_path, 'rb') as f:
                    reader = pa.ipc.RecordBatchStreamReader(f)
                    schema = reader.schema
                    batches = list(reader)
                    if batches:
                        table = pa.Table.from_batches(batches, schema)
                        print_table_info(table, max_rows=max_rows)
                        if show_stats:
                            print_statistics(table)
                        return table
            except Exception as e2:
                print(f"Could not read as Arrow file: {e2}")
                return None
    
    return None


def view_hf_dataset(dataset_path: Path, max_rows: int = 10, show_stats: bool = False):
    """View a HuggingFace dataset (backed by Arrow)."""
    if Dataset is None or load_from_disk is None:
        print("Error: HuggingFace datasets library is not installed.")
        print("Install it with: pip install datasets")
        return None
    
    print(f"\n{'=' * 80}")
    print(f"VIEWING HUGGINGFACE DATASET: {dataset_path}")
    print(f"{'=' * 80}")
    
    try:
        dataset = load_from_disk(str(dataset_path))
        print(f"Dataset type: HuggingFace Dataset")
        print(f"Number of rows: {len(dataset):,}")
        print(f"Number of columns: {len(dataset.column_names)}")
        
        print("\n" + "-" * 80)
        print("FEATURES/SCHEMA")
        print("-" * 80)
        for key, feature in dataset.features.items():
            print(f"  {key}: {feature}")
        
        print("\n" + "-" * 80)
        print(f"COLUMN NAMES ({len(dataset.column_names)} columns)")
        print("-" * 80)
        for i, name in enumerate(dataset.column_names):
            print(f"  [{i}] {name}")
        
        if max_rows > 0:
            print("\n" + "-" * 80)
            print(f"FIRST {min(max_rows, len(dataset))} ROWS (sample)")
            print("-" * 80)
            # Convert to Arrow table for display
            arrow_table = dataset._data.table.slice(0, min(max_rows, len(dataset)))
            print_table_info(arrow_table, max_rows=max_rows)
        
        if show_stats:
            # Convert full dataset to Arrow table for statistics
            arrow_table = dataset._data.table
            print_statistics(arrow_table)
        
        return dataset
    
    except Exception as e:
        print(f"Error loading HuggingFace dataset: {e}")
        print("Trying to read as regular Arrow file...")
        return view_arrow_file(dataset_path, max_rows=max_rows, show_stats=show_stats)

## View Arrow File

Set the path to your Arrow file below and run the cell.

In [5]:
# Configuration
file_path = Path("data/aloha/plug_insert1/train/data-00000-of-00001.arrow")  # Change this to your file path
max_rows = 10  # Number of sample rows to display (set to 0 to hide)
show_stats = False  # Set to True to show statistics for numeric columns

# View the file
table = view_arrow_file(file_path, max_rows=max_rows, show_stats=show_stats)


VIEWING ARROW FILE: data/aloha/plug_insert1/train/data-00000-of-00001.arrow
File size: 193.02 KB

File type: Parquet format

Error reading file: Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
Trying alternative methods...

TABLE INFORMATION
Number of rows: 741
Number of columns: 10
Total size: 190.41 KB

--------------------------------------------------------------------------------
SCHEMA
--------------------------------------------------------------------------------
Schema:
  [0] observation.state: fixed_size_list<item: float>[9] (nullable)
      value_type: float
  [1] observation.effort: fixed_size_list<item: float>[9] (nullable)
      value_type: float
  [2] observation.images.cam_low: struct<path: string, timestamp: float> (nullable)
  [3] observation.images.cam_left_wrist: struct<path: string, timestamp: float> (nullable)
  [4] action: fixed_size_list<item: float>[9] (nullable)
      value_type: float
  [5] episode_index: 

## Interactive Data Exploration

If you loaded a table or dataset above, you can explore it further here.

In [11]:
# Access the table (if you loaded an Arrow file)
if 'table' in locals() and table is not None:
    print(f"Table has {len(table)} rows and {len(table.column_names)} columns")
    print(f"Column names: {table.column_names}")
    
    # Example: Access a specific column
    # col = table['column_name']
    # print(col[:5])  # First 5 values

Table has 741 rows and 10 columns
Column names: ['observation.state', 'observation.effort', 'observation.images.cam_low', 'observation.images.cam_left_wrist', 'action', 'episode_index', 'frame_index', 'timestamp', 'next.done', 'index']


In [27]:
# Convert to pandas DataFrame for easier manipulation (optional)
# Note: This loads the entire dataset into memory
if 'table' in locals() and table is not None:
    try:
        import pandas as pd
        df = table.to_pandas()
    finally:
        pass

df.iloc[:,1]

0      [0.0, 31.0, 9.0, -354.0, -358.0, 19.0, -26.0, ...
1      [0.0, 49.0, 18.0, -343.0, -348.0, 20.0, -32.0,...
2      [0.0, 146.0, 130.0, -406.0, -292.0, 17.0, -44....
3      [0.0, 93.0, 69.0, -374.0, -344.0, 17.0, -45.0,...
4      [0.0, 52.0, 42.0, -295.0, -243.0, 17.0, -41.0,...
                             ...                        
736    [0.0, -248.0, -255.0, -295.0, -118.0, 0.0, -33...
737    [0.0, -248.0, -255.0, -295.0, -119.0, 0.0, -35...
738    [0.0, -245.0, -257.0, -295.0, -116.0, 0.0, -34...
739    [0.0, -252.0, -255.0, -295.0, -113.0, 0.0, -35...
740    [0.0, -247.0, -258.0, -295.0, -113.0, 0.0, -36...
Name: observation.effort, Length: 741, dtype: object